# Image Embeddings Visualization

Visualizes the **image** embedding space (512-d) using PCA.
Embeddings: `sentence-transformers/clip-ViT-B-32` → `image_products`

**Architecture:** Local Python generation → BigQuery storage → PCA visualization

**Kernel:** Select `Python (fakestore .venv)` in Jupyter

In [10]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')

PROJECT_ID = os.environ['BQ_PROJECT']
DATASET_VECTOR = os.environ.get('BQ_DATASET_VECTOR', 'fakestore_vector')

print(f'Project: {PROJECT_ID}')
print(f'Dataset: {DATASET_VECTOR}')

Project: academia-anahuac-mtiia
Dataset: fakestore_vector


In [11]:
import numpy as np
import pandas as pd
from google.cloud import bigquery
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Load Data from BigQuery

In [12]:
client = bigquery.Client(project=PROJECT_ID)

def load_embeddings(table):
    query = f"""
    SELECT
        product_id,
        title,
        category,
        price,
        rating_value,
        rating_count,
        image_url,
        embedding
    FROM `{PROJECT_ID}.{DATASET_VECTOR}.{table}`
    ORDER BY product_id
    """
    df = client.query(query).to_dataframe()
    emb = np.array(df['embedding'].tolist())
    print(f'{table}: {len(df)} products, {emb.shape[1]}-dimensional embeddings')
    return df, emb

df, emb = load_embeddings('image_products')
print(f'Categories: {df["category"].unique()}')

/Users/gus/trabajo/academia/potential-happiness/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


image_products: 20 products, 512-dimensional embeddings
Categories: <ArrowStringArray>
['men's clothing', 'jewelery', 'electronics', 'women's clothing']
Length: 4, dtype: str


---
## 2. PCA 2D — Image Embeddings

In [13]:
pca_2d = PCA(n_components=2)
emb_2d = pca_2d.fit_transform(emb)

df['pca_x'] = emb_2d[:, 0]
df['pca_y'] = emb_2d[:, 1]

print(f'PCA explained variance: {pca_2d.explained_variance_ratio_.sum():.2%}')
print(f'  PC1: {pca_2d.explained_variance_ratio_[0]:.2%}')
print(f'  PC2: {pca_2d.explained_variance_ratio_[1]:.2%}')

PCA explained variance: 35.94%
  PC1: 22.53%
  PC2: 13.41%


In [14]:
fig = px.scatter(
    df,
    x='pca_x', y='pca_y',
    color='category',
    hover_data=['product_id', 'title', 'price', 'rating_value'],
    title='Image Embeddings — PCA 2D (colored by Category)',
    labels={'pca_x': 'PC1', 'pca_y': 'PC2'},
    width=900, height=600
)
fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
fig.show()

In [15]:
fig = px.scatter(
    df,
    x='pca_x', y='pca_y',
    color='price',
    size='rating_count',
    hover_data=['product_id', 'title', 'category', 'price'],
    title='Image Embeddings — PCA 2D (colored by Price, sized by Rating Count)',
    labels={'pca_x': 'PC1', 'pca_y': 'PC2'},
    color_continuous_scale='Viridis',
    width=900, height=600
)
fig.update_traces(marker=dict(line=dict(width=1, color='white')))
fig.show()

---
## 3. PCA 3D — Image Embeddings

In [16]:
pca_3d = PCA(n_components=3)
emb_3d = pca_3d.fit_transform(emb)
df['pca_z'] = emb_3d[:, 2]
print(f'PCA 3D explained variance: {pca_3d.explained_variance_ratio_.sum():.2%}')

PCA 3D explained variance: 44.16%


In [17]:
fig_3d = px.scatter_3d(
    df,
    x='pca_x', y='pca_y', z='pca_z',
    color='category',
    hover_data=['product_id', 'title', 'price'],
    title='Image Embeddings — PCA 3D',
    labels={'pca_x': 'PC1', 'pca_y': 'PC2', 'pca_z': 'PC3'},
    width=900, height=700
)
fig_3d.update_traces(marker=dict(size=8, line=dict(width=1, color='white')))
fig_3d.show()

---
## 4. Explained Variance

In [18]:
pca_full = PCA().fit(emb)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100
n_comp = min(20, len(cumvar))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, n_comp + 1)), y=cumvar[:n_comp],
    name='Image Embeddings', mode='lines+markers'
))
fig.update_layout(
    title='Cumulative Variance Explained',
    xaxis_title='Principal Component',
    yaxis_title='Cumulative Variance (%)',
    width=800, height=500
)
fig.show()

---
## 5. Embedding Similarity Matrix

In [19]:
sim = cosine_similarity(emb)
short_titles = [t[:25] + '...' if len(t) > 25 else t for t in df['title']]

fig_sim = px.imshow(
    sim,
    x=short_titles, y=short_titles,
    color_continuous_scale='RdBu_r',
    title='Image Embedding Similarity Matrix (Cosine)',
    width=900, height=800
)
fig_sim.update_layout(
    xaxis={'tickangle': 45, 'tickfont': {'size': 8}},
    yaxis={'tickfont': {'size': 8}}
)
fig_sim.show()

---
## 6. Category Centroid Analysis

In [20]:
centroids = df.groupby('category')['embedding'].apply(
    lambda x: np.mean(np.array(x.tolist()), axis=0)
).to_dict()
cats = list(centroids.keys())
matrix = np.array([centroids[c] for c in cats])
sim_cat = cosine_similarity(matrix)

fig_centroid = px.imshow(
    sim_cat,
    x=cats, y=cats,
    color_continuous_scale='Blues',
    text_auto='.2f',
    title='Category Centroid Similarity (Image Embeddings)',
    width=600, height=500
)
fig_centroid.show()

---
## 7. Product vs Category Centroid Similarity

In [21]:
df['centroid_sim'] = df.apply(
    lambda row: cosine_similarity([row['embedding']], [centroids[row['category']]])[0][0],
    axis=1
)
fig = px.bar(
    df.sort_values('centroid_sim', ascending=True),
    x='centroid_sim', y='title',
    color='category',
    title='Product Similarity to Category Centroid — Image Embeddings',
    labels={'centroid_sim': 'Cosine Similarity', 'title': ''},
    width=800, height=600,
    orientation='h'
)
fig.show()